# Model 2 — Keras Push-up Classifier
Auto-labels frames from downloaded videos using elbow angle logic, then trains a neural network.
Exports `.keras`, scaler JSON, and `.tflite`.

In [1]:
%pip install mediapipe opencv-python numpy tensorflow scikit-learn kagglehub

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.1
[notice] To update, run: C:\Users\gusta\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


## Step 1 — Setup

In [2]:
import cv2
import mediapipe as mp
import numpy as np
import os
import json
import shutil
import urllib.request
from mediapipe.tasks import python as mp_python
from mediapipe.tasks.python import vision as mp_vision

MODEL_PATH = os.path.join(os.getcwd(), 'pose_landmarker.task')

if not os.path.exists(MODEL_PATH):
    print('Downloading pose landmarker model...')
    urllib.request.urlretrieve(
        'https://storage.googleapis.com/mediapipe-models/pose_landmarker/pose_landmarker_lite/float16/latest/pose_landmarker_lite.task',
        MODEL_PATH
    )
    print('Done.')
else:
    print(f'Model found: {MODEL_PATH}')

LEFT_SHOULDER  = 11
RIGHT_SHOULDER = 12
LEFT_ELBOW     = 13
RIGHT_ELBOW    = 14
LEFT_WRIST     = 15
RIGHT_WRIST    = 16

PUSHUP_CONFIG = {
    'down_elbow_threshold':  90,   # below = down position
    'up_elbow_threshold':   160,   # above = up position
    'visibility_threshold':  0.6,
}

def calculate_angle(a, b, c):
    a, b, c = np.array(a), np.array(b), np.array(c)
    ba, bc = a - b, c - b
    cosine = np.dot(ba, bc) / (np.linalg.norm(ba) * np.linalg.norm(bc) + 1e-6)
    return np.degrees(np.arccos(np.clip(cosine, -1.0, 1.0)))

def auto_label(landmarks):
    def get(idx):
        p = landmarks[idx]
        return [p.x, p.y], p.visibility
    sh_l, v1 = get(LEFT_SHOULDER)
    sh_r, v2 = get(RIGHT_SHOULDER)
    el_l, v3 = get(LEFT_ELBOW)
    el_r, v4 = get(RIGHT_ELBOW)
    wr_l, v5 = get(LEFT_WRIST)
    wr_r, v6 = get(RIGHT_WRIST)
    if min(v1,v2,v3,v4,v5,v6) < PUSHUP_CONFIG['visibility_threshold']:
        return -1, None
    elbow_l  = calculate_angle(sh_l, el_l, wr_l)
    elbow_r  = calculate_angle(sh_r, el_r, wr_r)
    avg_elbow = (elbow_l + elbow_r) / 2
    flat = [v for lm in landmarks for v in (lm.x, lm.y, lm.visibility)]
    if avg_elbow < PUSHUP_CONFIG['down_elbow_threshold']:
        return 1, flat   # down
    elif avg_elbow > PUSHUP_CONFIG['up_elbow_threshold']:
        return 0, flat   # up
    return -1, None      # transition — discard

print('Setup complete.')

Done.
Setup complete.


## Step 2 — Download training videos from Kaggle

In [ ]:
import kagglehub

VIDEOS_DIR = os.path.join(os.getcwd(), 'trainingData', 'videos')
os.makedirs(VIDEOS_DIR, exist_ok=True)

dataset_path = kagglehub.dataset_download('hasyimabdillah/workoutfitness-video')
print('Downloaded to:', dataset_path)

pushup_folder = os.path.join(dataset_path, 'push-up')
print('Push-up folder:', pushup_folder)

for fname in os.listdir(pushup_folder):
    if fname.endswith(('.mp4', '.mov', '.avi')):
        src = os.path.join(pushup_folder, fname)
        dst = os.path.join(VIDEOS_DIR, fname)
        if not os.path.exists(dst):
            shutil.copy2(src, dst)
            print(f'Copied: {fname}')
        else:
            print(f'Already exists: {fname}')

print(f'\nTotal videos ready: {len(os.listdir(VIDEOS_DIR))}')

## Step 3 — Extract and auto-label frames

In [4]:
VIDEO_PATHS = [
    os.path.join(VIDEOS_DIR, f)
    for f in os.listdir(VIDEOS_DIR)
    if f.endswith(('.mp4', '.mov', '.avi'))
]
print(f'Found {len(VIDEO_PATHS)} videos')

def extract_and_label(video_path):
    X, y = [], []
    cap = cv2.VideoCapture(video_path)
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    base_options = mp_python.BaseOptions(model_asset_path=MODEL_PATH)
    options = mp_vision.PoseLandmarkerOptions(
        base_options=base_options,
        running_mode=mp_vision.RunningMode.VIDEO,
    )
    timestamp_ms = 0
    processed = 0
    with mp_vision.PoseLandmarker.create_from_options(options) as landmarker:
        while cap.isOpened():
            ret, frame = cap.read()
            if not ret:
                break
            timestamp_ms += int(1000 / 30)
            rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb)
            result = landmarker.detect_for_video(mp_image, timestamp_ms)
            if result.pose_landmarks:
                label, flat = auto_label(result.pose_landmarks[0])
                if label != -1:
                    X.append(flat)
                    y.append(label)
            processed += 1
            if processed % 100 == 0:
                print(f'  {processed}/{total} frames')
    cap.release()
    return X, y

X_all, y_all = [], []
for path in VIDEO_PATHS:
    print(f'Processing {os.path.basename(path)}...')
    X_v, y_v = extract_and_label(path)
    X_all.extend(X_v)
    y_all.extend(y_v)

X = np.array(X_all, dtype=np.float32)
y = np.array(y_all, dtype=np.int32)
print(f'\nDataset: {len(X)} frames — up: {np.sum(y==0)}, down: {np.sum(y==1)}')

Found 56 videos
Processing push-up_1.mp4...
  100/150 frames
Processing push-up_10.mp4...
  100/150 frames
Processing push-up_11.mp4...
  100/150 frames
Processing push-up_12.mp4...
  100/150 frames
Processing push-up_13.mp4...
  100/150 frames
Processing push-up_14.mp4...
  100/150 frames
Processing push-up_15.mp4...
Processing push-up_16.mp4...
  100/140 frames
Processing push-up_17.mp4...
Processing push-up_18.mp4...
Processing push-up_19.mp4...
  100/150 frames
Processing push-up_2.mp4...
  100/150 frames
Processing push-up_20.mp4...
  100/150 frames
Processing push-up_21.mp4...
  100/150 frames
Processing push-up_22.mp4...
  100/150 frames
Processing push-up_23.mp4...
  100/116 frames
Processing push-up_24.mp4...
  100/126 frames
Processing push-up_25.mp4...
Processing push-up_26.mp4...
  100/150 frames
Processing push-up_27.mp4...
  100/150 frames
Processing push-up_28.mp4...
  100/113 frames
Processing push-up_29.mp4...
  100/150 frames
Processing push-up_3.mp4...
  100/150 fram

## Step 4 — Train

In [5]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import tensorflow as tf
from tensorflow import keras

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test  = scaler.transform(X_test)

model = keras.Sequential([
    keras.layers.Input(shape=(99,)),
    keras.layers.Dense(128, activation='relu'),
    keras.layers.Dropout(0.3),
    keras.layers.Dense(64, activation='relu'),
    keras.layers.Dropout(0.3),
    keras.layers.Dense(1, activation='sigmoid'),
])

model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
model.summary()

history = model.fit(X_train, y_train, validation_data=(X_test, y_test), epochs=30, batch_size=32)

loss, acc = model.evaluate(X_test, y_test)
print(f'Test accuracy: {acc:.3f}')

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 128)            │        12,800 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 21,121 (82.50 KB)

 Trainable params: 21,121 (82.50 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/30
73/73 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.9223 - loss: 0.2055 - val_accuracy: 0.9879 - val_loss: 0.0378
Epoch 2/30
73/73 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.9918 - loss: 0.0363 - val_accuracy: 0.9948 - val_loss: 0.0143
Epoch 3/30
73/73 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.9944 - loss: 0.0189 - val_accuracy: 1.0000 - val_loss: 0.0064
Epoch 4/30
73/73 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.9948 - loss: 0.0150 - val_accuracy: 0.9983 - val_loss: 0.0133
Epoch 5/30
73/73 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.9957 - loss: 0.0133 - val_accuracy: 1.0000 - val_loss: 0.0034
Epoch 6/30
73/73 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.9961 - loss: 0.0140 - val_accuracy: 1.0000 - val_loss: 0.0019
Epoch 7/30
73/73 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.9953 - loss: 0.0114 - val_accuracy: 1.0000 - val_loss: 0.0022
Epoch 8/30
73/73 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.9974 - loss: 0.0083 - val_accuracy: 1.0000 - val_loss:

## Step 5 — Export

In [6]:
model.save('pushup_model.keras')
print('Exported: pushup_model.keras')

scaler_params = {'mean': scaler.mean_.tolist(), 'scale': scaler.scale_.tolist()}
with open('pushup_model2_scaler.json', 'w') as f:
    json.dump(scaler_params, f)
print('Exported: pushup_model2_scaler.json')

converter = tf.lite.TFLiteConverter.from_keras_model(model)
tflite_model = converter.convert()
with open('pushup_model2.tflite', 'wb') as f:
    f.write(tflite_model)
print('Exported: pushup_model2.tflite')

Exported: pushup_model.keras
Exported: pushup_model2_scaler.json
INFO:tensorflow:Assets written to: C:\Users\gusta\AppData\Local\Temp\tmpn3gw7k0u\assets


INFO:tensorflow:Assets written to: C:\Users\gusta\AppData\Local\Temp\tmpn3gw7k0u\assets


Saved artifact at 'C:\Users\gusta\AppData\Local\Temp\tmpn3gw7k0u'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 99), dtype=tf.float32, name='keras_tensor')
Output Type:
  TensorSpec(shape=(None, 1), dtype=tf.float32, name=None)
Captures:
  2395421512080: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2395421513424: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2395421511888: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2395421513040: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2395421513232: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2395421513808: TensorSpec(shape=(), dtype=tf.resource, name=None)
Exported: pushup_model2.tflite


## Step 6 — Live inference

In [7]:
THRESHOLD = 0.5

base_options = mp_python.BaseOptions(model_asset_path=MODEL_PATH)
options = mp_vision.PoseLandmarkerOptions(
    base_options=base_options,
    running_mode=mp_vision.RunningMode.VIDEO,
)

cap = cv2.VideoCapture(0)
pushup_count = 0
in_down = False
timestamp_ms = 0

with mp_vision.PoseLandmarker.create_from_options(options) as landmarker:
    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break
        timestamp_ms += int(1000 / 30)
        rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb)
        result = landmarker.detect_for_video(mp_image, timestamp_ms)
        if result.pose_landmarks:
            flat = [v for lm in result.pose_landmarks[0] for v in (lm.x, lm.y, lm.visibility)]
            prob = model.predict(scaler.transform([flat]), verbose=0)[0][0]
            is_down = prob > THRESHOLD
            if is_down and not in_down:
                in_down = True
            elif not is_down and in_down:
                pushup_count += 1
                in_down = False
            label = f'DOWN {prob:.2f}' if is_down else f'UP {1-prob:.2f}'
            color = (0, 255, 255) if is_down else (0, 255, 0)
            cv2.putText(frame, f'{label}  Reps: {pushup_count}', (20, 50),
                        cv2.FONT_HERSHEY_SIMPLEX, 1.1, color, 2)
        cv2.imshow('Push-up Detector - Model 2', frame)
        if cv2.waitKey(1) & 0xFF == ord('q'):
            break

cap.release()
cv2.destroyAllWindows()